## Loading the Data

In [91]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn xgboost streamlit


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip3.13 install --upgrade pip


In [92]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [93]:
# Read the data, lines set to true so that it is understood that each line is own JSON object

portfolio = pd.read_json('data/portfolio.json', lines = True)
profile = pd.read_json('data/profile.json', lines=True)
transcript = pd.read_json('data/transcript.json', lines=True)

With our datasets loaded in, let's now look at the structure of data and see if we can extract anything interesting.

In [94]:
print(portfolio.shape)
portfolio.info()
portfolio.describe()

(10, 6)
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   reward      10 non-null     int64 
 1   channels    10 non-null     object
 2   difficulty  10 non-null     int64 
 3   duration    10 non-null     int64 
 4   offer_type  10 non-null     str   
 5   id          10 non-null     str   
dtypes: int64(3), object(1), str(2)
memory usage: 1006.0+ bytes


,reward,difficulty,duration
count,10.000000,10.000000,10.000000
mean,4.200000,7.700000,6.500000
std,3.583915,5.831905,2.321398
min,0.000000,0.000000,3.000000
25%,2.000000,5.000000,5.000000
50%,4.000000,8.500000,7.000000
75%,5.000000,10.000000,7.000000
max,10.000000,20.000000,10.000000


In [95]:
print(profile.shape)
profile.info()
profile.describe()

(17000, 5)
<class 'pandas.DataFrame'>
RangeIndex: 17000 entries, 0 to 16999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            14825 non-null  str    
 1   age               17000 non-null  int64  
 2   id                17000 non-null  str    
 3   became_member_on  17000 non-null  int64  
 4   income            14825 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 1.2 MB


,age,became_member_on,income
count,17000.000000,1.700000e+04,14825.000000
mean,62.531412,2.016703e+07,65404.991568
std,26.738580,1.167750e+04,21598.299410
min,18.000000,2.013073e+07,30000.000000
25%,45.000000,2.016053e+07,49000.000000
50%,58.000000,2.017080e+07,64000.000000
75%,73.000000,2.017123e+07,80000.000000
max,118.000000,2.018073e+07,120000.000000


We can see that the max age is 118, and that the income count is only 14825.

In [96]:
print(transcript.shape)
transcript.info()
transcript.describe()

(306534, 4)
<class 'pandas.DataFrame'>
RangeIndex: 306534 entries, 0 to 306533
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   person  306534 non-null  str   
 1   event   306534 non-null  str   
 2   value   306534 non-null  object
 3   time    306534 non-null  int64 
dtypes: int64(1), object(1), str(2)
memory usage: 22.3+ MB


,time
count,306534.000000
mean,366.382940
std,200.326314
min,0.000000
25%,186.000000
50%,408.000000
75%,528.000000
max,714.000000


## Data Analysis


In [97]:
# Let's look at the value distributions for some cateogrical columns:

print(profile['gender'].value_counts())
print()
print(transcript['event'].value_counts())
print()
print(portfolio['offer_type'].value_counts())

gender
M    8484
F    6129
O     212
Name: count, dtype: int64

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

offer_type
bogo             4
discount         4
informational    2
Name: count, dtype: int64


The gender breakdown is M: 8484, F: 6129, O: 212, with 2175 null values (those are the age = 118 rows, the same customers with missing demographics).

The transcript has 4 event types. 76,277 offers were received, but only 33,579 were completed: roughly a 44% raw completion rate. This will be our target variable. Note that 138,953 transactions exist separately: customers spending money independent of any offer.

The portfolio has 4 BOGO, 4 discount, and 2 informational offers. Informational offers have no reward and can't technically be "completed". We'll need to handle those separately.

In [98]:
# Let's peek at the value column in the transcript, where offer IDs are buried.
transcript['value'].sample(20)

157224     {'offer id': '5a8bc65990b245e5a138643cd4eb9837'}
99810                        {'amount': 0.8200000000000001}
112048     {'offer id': '3f207df678b143eea3cee63160fa8bed'}
257825     {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
231979                                     {'amount': 1.06}
107548                                    {'amount': 20.68}
145651                                    {'amount': 18.88}
32702      {'offer id': '5a8bc65990b245e5a138643cd4eb9837'}
51753     {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
276393                                     {'amount': 1.06}
149578                                     {'amount': 2.49}
166137                                    {'amount': 16.35}
156406     {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
80703                                      {'amount': 1.41}
119834     {'offer id': '3f207df678b143eea3cee63160fa8bed'}
66621                                      {'amount': 4.68}
102882    {'offer_id': 'fafdcd668e3743c1

We see that there are a few different data types:

{'offer id': '...'} : note the space in the key (most offer rows)

{'offer_id': '...'} : note the underscore (row 186219. This is a completed offer event)

{'amount': 22.79} — transaction rows

Let's see if we can detect a pattern on when the underscore occurs:

In [99]:
transcript[transcript['event'] == 'offer completed']['value'].head(15)

12658    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
12672    {'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4...
12679    {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
12692    {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...
12697    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
12717    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
12721    {'offer_id': '2298d6c36e964ae4a3e7e9706d1fb8c2...
12744    {'offer_id': 'f19421c1d4aa40978ebb69ca19b0e20d...
12764    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
12767    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0...
12780    {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
12784    {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9...
12786    {'offer_id': '2906b810c7d4411798c6938adc9daaa5...
12798    {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd...
12817    {'offer_id': '2298d6c36e964ae4a3e7e9706d1fb8c2...
Name: value, dtype: object

We observe that every single offer completed row uses offer_id (underscore), while offer received and offer viewed rows use offer id (space). When we later extract offer IDs, we will need to handle such cases using this information.

A few other quirks to note:

- The age max of 118 is a confirmed missing marker

- became_member_on shows as 2.016703e+07 — that's just scientific notation for 20170212. It's being treated as a number, instead of a date. We'll convert it to an actual date in cleaning.

- income ranges from 30k to 120k, mean ~65k, which is reasonable, with no obvious outliers there.


## Data Cleaning

The current issues to fix:

- Remove rows where age == 118 (missing demographic marker)
- Convert became_member_on from integer to datetime
- Standardize value column keys (offer id → offer_id)


In [100]:
# We will first confirm that the a row with age 118 indicates a dead row:

profile[profile['age'] == 118][['age', 'gender', 'income']].head(10)

,age,gender,income
0,118,NaN,NaN
2,118,NaN,NaN
4,118,NaN,NaN
6,118,NaN,NaN
7,118,NaN,NaN
9,118,NaN,NaN
10,118,NaN,NaN
11,118,NaN,NaN
17,118,NaN,NaN
23,118,NaN,NaN


In [101]:
# Confirmed. 

profile = profile[profile['age'] != 118]
print(profile.shape)

# Now profile has the valid number of customers with valid demographics.

(14825, 5)


In [102]:
# Let's now convert became_member_on to a datetime.

profile['became_member_on'] = pd.to_datetime(profile['became_member_on'], format = '%Y%m%d')
profile['became_member_on'].head()

1    2017-07-15
3    2017-05-09
5    2018-04-26
8    2018-02-09
12   2017-11-11
Name: became_member_on, dtype: datetime64[us]

In [106]:
# Let's now create a function to standardize the value column keys. 
# Below will replace offer id with offer_id

def standardize_value(d):
    if not isinstance(d, dict):
        return d

    if 'offer id' in d:
        d['offer_id'] = d.pop('offer id') # pop removes key and returns value
    return d


# Let's now apply the function across each row in transcript:

transcript['value'] = transcript['value'].apply(standardize_value)
transcript['value'].sample(15)

281732                                    {'amount': 23.0}
180131    {'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}
53563     {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd'}
41969                                    {'amount': 12.85}
175372                                    {'amount': 0.76}
80558     {'offer_id': '2906b810c7d4411798c6938adc9daaa5'}
40884                                    {'amount': 40.15}
27354                                     {'amount': 3.17}
165257    {'offer_id': '5a8bc65990b245e5a138643cd4eb9837'}
76407     {'offer_id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}
23472     {'offer_id': '2906b810c7d4411798c6938adc9daaa5'}
270587                                   {'amount': 21.43}
199914                                   {'amount': 27.61}
82221     {'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd'}
256603    {'offer_id': '5a8bc65990b245e5a138643cd4eb9837'}
Name: value, dtype: object

## Feature Engineering

We want to combine these tables into one, flat table where each row is one (customer, offer) pair, with features that we can feed into a model. Right now the data is split acrosss three separate tables, so we need to join them:

To build this, we will need to: 

- Extract offer events from transcript: pull out each offer received, offer viewed, offer completed row with its offer_id

- Determine if an offer was completed: for each (customer, offer) pair, was there a corresponding offer completed event?

- Join with profile: attach the customer's demographics

- Join with portfolio: attach the offer's details

- Engineer member_tenure_days — how long had they been a member when the offer was sent?


In [ ]:
offer_events = transcript[transcript['event'] != 'transaction'].copy()
offer_events['offer_id'] = offer_events['value'].apply(lambda d: d.get('offer_id'))
offer_events.head()

,person,event,value,time,offer_id
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0,9b98b8c7a33c4b65b9aebfe6a799e6d9
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer_id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0,0b1e1539f2cc45b7b9fa7c272da2e1d7
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer_id': '2906b810c7d4411798c6938adc9daaa5'},0,2906b810c7d4411798c6938adc9daaa5
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4'},0,fafdcd668e3743c1bb461111dcafc2a4
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0,4d5c57ea9a6940dd891ad53e9dbe8da0


In [108]:
# We now extracted the offer id to its own column. Next, we will build the target varaible.
# We ask the question, for each unique (customer, offer) pair, did a offer completed event exist?

completed = offer_events[offer_events['event'] == 'offer completed'][['person', 'offer_id']].drop_duplicates()
completed['completed'] = 1 # for binary labeling
completed.head()

,person,offer_id,completed
12658,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,2906b810c7d4411798c6938adc9daaa5,1
12672,fe97aa22dd3e48c8b143116a8403dd52,fafdcd668e3743c1bb461111dcafc2a4,1
12679,629fc02d56414d91bca360decdfa9288,9b98b8c7a33c4b65b9aebfe6a799e6d9,1
12692,676506bad68e4161b9bbaffeb039626b,ae264e3637204a6fb9bb56bc8210ddfd,1
12697,8f7dd3b2afe14c078eb4f6e6fe4ba97d,4d5c57ea9a6940dd891ad53e9dbe8da0,1
